# Prostate Patient_1 — ground-truth check of UNI2 A8 (2026.11.16)

The prostate notebooks only showed predictions. Erickson et al. (Nature 2022) ship two references per Visium section in `Patient_1/Visium_with_annotation/{section}/`, already downloaded by `download_patient1_visium_annotation.py`:

* `{section}_Final_Consensus_Annotations.csv` — pathologist consensus label for every spot (Gleason grade groups, benign, PIN, stroma, inflammation, ...).
* `filtered_feature_bc_matrix.h5` — the spot RNA.

This notebook places each StarDist nucleus in its 55 µm capture disc and reports, for every annotated section:

1. mean Hist2Pheno compartment fraction per annotated region (e.g. immune fraction inside "inflammation" spots vs elsewhere);
2. one-vs-rest **AUROC** of the matching Hist2Pheno fraction for each region (0.5 = no signal);
3. Spearman of each Hist2Pheno compartment with its marker-gene score (MS4A1/CD79A for B, CD3D/CD3E for T, AMACR/ERG for tumour, ...).

Annotation strings are mapped to broad regions by `visium_validation.annotation_region`; the raw label counts are printed so the mapping can be checked.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
import os
os.environ["UNI2_CKPT"] = str(REPO / "data/PanCancer/result_uni2_A8_l12_balanced_paired_v2_seed42/cross_dataset_cv/A8_l12_balanced/best_mlp_gpu.pt")
STAR_PROB_MIN = 0.0  # 2026.10.28, set above 0 to store a separate _probXX folder
import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0" if float(STAR_PROB_MIN) <= 0 else f"{float(STAR_PROB_MIN):.2f}"
os.environ["UNI2_PRED_DIRNAME"] = "pred_uni2_A8_bestmlp_paired_v2_seed42"
sys.path.insert(0, str(REPO / "code" / "External_val" / "Prostate"))

import prostate_external as prostate_ext
prostate_ext.use_embedder("uni2")
PRED_ROOT = prostate_ext.PRED_ROOT
composition_table = prostate_ext.composition_table
discover_samples = prostate_ext.discover_samples
geometry = prostate_ext.geometry
sample_paths = prostate_ext.sample_paths
ensure_l2_probs = prostate_ext.ensure_l2_probs
plot_spatial = prostate_ext.plot_spatial
plot_composition = prostate_ext.plot_composition

PATIENT = "Patient_1"
SERIES = "all"
# 2026.11.16, every Patient_1 section that has a consensus annotation
ANNOTATED = ("H1_2", "H1_4", "H1_5", "H2_1", "H2_2", "H2_5", "V1_2")
group = [s for s in discover_samples(PATIENT) if s.split("__", 1)[-1] in ANNOTATED]
fig_dir = PRED_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
# Re-run the plot cell after changing these. Notebook display uses DPI; saved PNGs use SAVE_DPI.
SAVE_FIGURES = False
FIGSIZE = (17.2, 6.6)
DPI = 160
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 1.2
print(f"{PATIENT} {SERIES}  n={len(group)}")
for sample in group:
    g = geometry(sample)
    section = sample.split("__", 1)[-1]
    print(
        f"{section}  mpp={g['mpp_um']:.4f} µm/px  "
        f"native14={g['patch_um_if_native_16px']:.2f} µm  "
        f"after rescale={g['patch_um_after_rescale']:.1f} µm  "
        f"scale={g['scale']:.4f}  ({g['mpp_source']})"
    )
    pth = sample_paths(sample)
    print(f"  pred exists: {pth['pred'].is_file()}  {pth['pred']}")

# 2026.10.28, mandatory current-input preflight / 后续直接读 h5ad 的图表也不得绕过上游
import os as _pipeline_os
_pipeline_os.environ["STAR_PROB_MIN"] = str(globals().get("STAR_PROB_MIN", 0.0))
if "ensure_l2_probs" in globals():
    _owner = ensure_l2_probs.__self__
    _current_samples = list(group)
    _owner.cmd_predict(_current_samples, force=False, allow_cpu=True)
elif "ue" in globals():
    ue.cmd_predict(COHORT, samples, cuda_device=_pipeline_os.environ.get("CUDA_DEVICE", "0"),
                   force=False, allow_cpu=True, embedder=EMBEDDER)
else:
    raise RuntimeError("No current prediction validator; run the setup cell first")

In [ ]:
# 2026.11.16, consensus annotation + RNA marker validation
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "External_val" / "NPC"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
import visium_validation as vv
importlib.reload(vv)
from npc_figures import load_aligned_l2_probs

MARKER_PAIRS = {f"frac_{g}": g for g in ("B", "Plasma", "T", "Myeloid", "Stroma", "Endothelial", "Epithelial")}
MARKER_PAIRS["frac_Tumor"] = "Tumor"
MARKER_SETS = dict(vv.MARKERS, Tumor=vv.TUMOR_MARKERS["Prostate"])

per_region, per_auc, per_marker, raw_labels = [], [], [], []
for sample in group:
    section = sample.split("__", 1)[-1]
    paths = sample_paths(sample)
    vis = paths["visium"]
    ann_path = vis / f"{section}_Final_Consensus_Annotations.csv"
    if not ann_path.is_file():
        print(f"[skip] {sample}: {ann_path} missing")
        continue
    spots = vv.read_tissue_positions(vis)
    radius = vv.capture_radius_px(paths["scalefactors"])
    ensure_l2_probs(sample)
    xy, probs, class_names = load_aligned_l2_probs(paths["pred"], paths["pred"].with_name(f"{sample}_l2_probs.npz"))
    h2p = vv.spot_h2p(xy, probs, class_names, spots, radius)

    labels = vv.read_annotations(ann_path)
    regions = labels.map(vv.annotation_region)
    raw_labels.append(pd.DataFrame({"section": section, "label": labels.to_numpy(), "region": regions.to_numpy()}))
    mean, auc = vv.annotation_report(h2p, regions)
    per_region.append(mean.assign(section=section).reset_index())
    per_auc.append(auc.assign(section=section))

    h5 = vis / "filtered_feature_bc_matrix.h5"
    if h5.is_file():
        ms = vv.marker_scores(h5, spots["barcode"], MARKER_SETS)
        ms.index = range(len(ms))
        per_marker.append(vv.agreement(h2p, ms, MARKER_PAIRS).assign(section=section))
    print(f"{section}: spots {len(spots):,}  radius {radius:.1f} px  nuclei/spot median "
          f"{h2p.n_nuclei[h2p.n_nuclei > 0].median():.0f}  annotated {int(regions.reindex(h2p.barcode).notna().sum()):,}")

raw = pd.concat(raw_labels)
print("\nannotation label -> region (check the mapping)")
display(raw.groupby(["label", "region"]).size().rename("n_spots").reset_index().sort_values("n_spots", ascending=False))

In [ ]:
region_tab = pd.concat(per_region)
auc_tab = pd.concat(per_auc)
print("mean Hist2Pheno compartment fraction per annotated region (pooled over sections)")
pooled = region_tab.drop(columns="section").groupby("region").apply(
    lambda d: pd.Series(np.average(d.drop(columns=["n_spots"]), axis=0, weights=d["n_spots"]),
                        index=d.drop(columns=["n_spots"]).columns)).round(3)
pooled.insert(0, "n_spots", region_tab.groupby("region")["n_spots"].sum())
display(pooled)
print("one-vs-rest AUROC of the matching Hist2Pheno fraction (0.5 = no signal)")
display(auc_tab.pivot_table(index=["region", "h2p_groups"], columns="section", values="auroc").round(3))
if per_marker:
    mk = pd.concat(per_marker)
    print("Spearman: Hist2Pheno compartment vs marker-gene score per spot")
    display(mk.pivot_table(index="reference", columns="section", values="spearman").round(3))
    mk.to_csv(fig_dir / f"{PATIENT}_h2p_vs_markers.csv", index=False)
region_tab.to_csv(fig_dir / f"{PATIENT}_h2p_by_annotation.csv", index=False)
auc_tab.to_csv(fig_dir / f"{PATIENT}_h2p_annotation_auroc.csv", index=False)

import matplotlib.pyplot as plt
show = pooled.drop(columns="n_spots")
fig, ax = plt.subplots(figsize=(1.0 + 0.7 * show.shape[1], 0.5 + 0.45 * show.shape[0]), dpi=DPI)
im = ax.imshow(show.to_numpy(), cmap="viridis", vmin=0, vmax=max(0.01, float(np.nanmax(show.to_numpy()))))
ax.set_xticks(range(show.shape[1]), show.columns, rotation=45, ha="right")
ax.set_yticks(range(show.shape[0]), [f"{r} (n={int(n)})" for r, n in zip(show.index, pooled["n_spots"])])
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        ax.text(j, i, f"{show.iat[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(im, ax=ax, label="mean Hist2Pheno fraction")
ax.set_title(f"{PATIENT}: Hist2Pheno composition by pathologist region")
fig.tight_layout()
fig.savefig(fig_dir / f"{PATIENT}_h2p_by_annotation.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

### How to read this

* A working model shows high `T`/`B` in **Immune** regions, high `Tumor` in **Tumor** regions and high `Stroma` in **Stroma** regions, with AUROC well above 0.5.
* Patient_1 H2_5 currently predicts 61% Immune / 42% B over the whole slide. If the Immune-region AUROC is high but the Immune fraction is also high in Tumor and Stroma regions, the model over-calls lymphocytes everywhere (calibration / domain shift), not a coordinate bug.
* If even Tumor vs Stroma is near 0.5, check registration (`spatial_HE` vs Visium pixel frame) before anything else.